# 04 · GARCH / Volatilidad Condicional — Finance Concept

**Contexto:** Robert Engle (Nobel 2003) observó que la varianza de los errores no es constante — aparece en **clusters**: períodos de alta volatilidad siguen a períodos de alta volatilidad. El modelo GARCH captura esta heterocedasticidad condicional con solo 3 parámetros.

**Campo de origen:** Econometría financiera · Engle (1982) ARCH · Bollerslev (1986) GARCH  
**Dataset:** Tipo de cambio PEN/USD — Banco Central de Reserva del Perú (BCRP)  
**API pública:** `https://estadisticas.bcrp.gob.pe/estadisticas/series/api/` — sin autenticación

---

## Marco teórico

### GARCH(1,1) — Bollerslev (1986)

$$r_t = \mu + \epsilon_t \qquad \epsilon_t = \sigma_t z_t \qquad z_t \sim \mathcal{N}(0,1)$$

$$\sigma_t^2 = \omega + \alpha \cdot \epsilon_{t-1}^2 + \beta \cdot \sigma_{t-1}^2$$

| Parámetro | Rol | Restricción |
|-----------|-----|-------------|
| $\omega > 0$ | Varianza base | Positivo |
| $\alpha \geq 0$ | Impacto del shock (ARCH) | No negativo |
| $\beta \geq 0$ | Persistencia (GARCH) | No negativo |
| $\alpha + \beta < 1$ | Estacionariedad | **Crítico** |

### EGARCH — Nelson (1991)

$$\ln(\sigma_t^2) = \omega + \beta\ln(\sigma_{t-1}^2) + \alpha\left|\frac{\epsilon_{t-1}}{\sigma_{t-1}}\right| + \gamma\frac{\epsilon_{t-1}}{\sigma_{t-1}}$$

$\gamma < 0$: depreciaciones del PEN generan más volatilidad que apreciaciones (efecto leverage).

### Varianza incondicional y forecast h-pasos

$$\bar{\sigma}^2 = \frac{\omega}{1-\alpha-\beta} \qquad \sigma_{t+h|t}^2 = \bar{\sigma}^2 + (\alpha+\beta)^{h-1}(\sigma_{t+1}^2 - \bar{\sigma}^2)$$

**Referencias:** Engle, R.F. (1982). *Econometrica* 50(4). Bollerslev, T. (1986). *Journal of Econometrics* 31(3). Nelson, D.B. (1991). *Econometrica* 59(2).

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from scipy.optimize import minimize
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    price='#1E293B', vol='#DC2626',   garch='#2563EB',
    fill='#DBEAFE',  green='#15803D', orange='#F59E0B',
    neutral='#94A3B8', egarch='#7C3AED'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS — BCRP Tipo de Cambio PEN/USD ──────────────────────────────────────
# Fuente: Banco Central de Reserva del Perú
# Serie: PD04637PD — Tipo de cambio venta (S/. por USD), frecuencia diaria
# API: https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{serie}/{formato}/{inicio}/{fin}
#
# OPCIÓN A — Descarga real (requiere internet):
#   import requests
#   url = 'https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PD04637PD/json/2018-1/2024-12'
#   data = requests.get(url).json()
#
# OPCIÓN B — Simulación calibrada con estadísticos reales del PEN/USD 2018-2024:
#   Media retorno diario ≈ 0.0001 (ligera tendencia depreciación)
#   Vol diaria ≈ 0.28%  (períodos tranquilos: 0.10-0.15%, crisis: 0.50-1.20%)
#   Clusters documentados: COVID Mar2020, incertidumbre electoral Jun2021,
#                           crisis política Nov2022
#   α ≈ 0.08, β ≈ 0.89 (alta persistencia — típico de FX emergentes)

try:
    import urllib.request
    url = ('https://estadisticas.bcrp.gob.pe/estadisticas/series/api/'
           'PD04637PD/json/2018-1/2024-12/ing')
    with urllib.request.urlopen(url, timeout=8) as r:
        raw = json.loads(r.read())
    records = raw['periods']
    dates  = pd.to_datetime([p['name'] for p in records], dayfirst=True, errors='coerce')
    values = pd.to_numeric([p['values'][0] for p in records], errors='coerce')
    tc = pd.Series(values.values, index=dates).dropna().sort_index()
    tc = tc[tc > 2.5]  # filtrar valores inválidos
    returns = tc.pct_change().dropna()
    SOURCE = 'BCRP API (datos reales)'
    print(f'✓ {SOURCE}: {len(tc)} observaciones')

except Exception as e:
    print(f'API no disponible ({e}) — usando simulación calibrada')
    SOURCE = 'Simulación calibrada (estadísticos reales BCRP 2018-2024)'

    n = 1500
    dates = pd.bdate_range('2018-01-02', periods=n)

    # Simular retornos con GARCH real del PEN/USD
    omega_r, alpha_r, beta_r = 0.0000008, 0.08, 0.89
    sigma2 = omega_r / (1 - alpha_r - beta_r)
    rets, sig2s = [], []

    # Eventos de alta volatilidad documentados
    high_vol_periods = [
        (pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-30'), 4.0),  # COVID
        (pd.Timestamp('2021-06-01'), pd.Timestamp('2021-09-30'), 3.0),  # elecciones
        (pd.Timestamp('2022-10-01'), pd.Timestamp('2023-01-31'), 2.5),  # crisis política
    ]

    for i, d in enumerate(dates):
        multiplier = 1.0
        for start, end, mult in high_vol_periods:
            if start <= d <= end:
                multiplier = mult
                break
        shock = np.random.normal(0, np.sqrt(sigma2 * multiplier))
        ret   = 0.00008 + shock
        rets.append(ret)
        sig2s.append(sigma2)
        sigma2 = omega_r + alpha_r * shock**2 + beta_r * sigma2

    returns = pd.Series(rets, index=dates)
    tc_vals = 3.25 * np.cumprod(1 + np.array(rets))
    tc = pd.Series(tc_vals, index=dates)

print(f'Fuente  : {SOURCE}')
print(f'Período : {returns.index[0].date()} → {returns.index[-1].date()}')
print(f'n       : {len(returns)} observaciones')
print(f'μ ret   : {returns.mean():.6f} ({returns.mean()*252:.2%} anual)')
print(f'σ ret   : {returns.std():.6f} ({returns.std()*np.sqrt(252):.2%} anual)')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Estadísticos clave ─────────────────────────────────────────────
r = returns
print(f'{"Métrica":<28} {"Valor":<16} Nota')
print('─' * 72)
rows = [
    ('n observaciones',       len(r),                         ''),
    ('TC inicial (S/.)',       f'{tc.iloc[0]:.4f}',           ''),
    ('TC final (S/.)',         f'{tc.iloc[-1]:.4f}',          ''),
    ('Depreciación total',     f'{(tc.iloc[-1]/tc.iloc[0]-1):.2%}', ''),
    ('Retorno μ diario',       f'{r.mean():.6f}',             f'{r.mean()*252:.2%} anual'),
    ('Vol σ diaria',           f'{r.std():.6f}',              f'{r.std()*np.sqrt(252):.2%} anual'),
    ('Skewness',               f'{r.skew():.3f}',             'negativo → depreciaciones más extremas'),
    ('Kurtosis exceso',        f'{r.kurt():.3f}',             '> 0 → fat tails'),
    ('VaR 1% diario',          f'{r.quantile(0.01):.4%}',    'pérdida máxima al 99%'),
    ('Max retorno diario',     f'{r.max():.4%}',              '(apreciación máxima)'),
    ('Min retorno diario',     f'{r.min():.4%}',              '(depreciación máxima)'),
    ('% días positivos',       f'{(r>0).mean():.1%}',         ''),
]
for label, val, note in rows:
    print(f'{label:<28} {str(val):<16} {note}')

In [ ]:
# ── EDA 2/2 — TC + retornos + clusters de volatilidad ────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1, 1], 'hspace': 0.06})
fig.suptitle(f'PEN/USD — Tipo de Cambio BCRP\n{SOURCE}', fontsize=11)

axes[0].plot(tc.index, tc.values, color=C['price'], lw=0.9)
axes[0].set_ylabel('S/. por USD')
axes[0].set_title('Tipo de cambio venta', loc='left', fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

col_r = np.where(returns >= 0, C['green'], C['vol'])
axes[1].bar(returns.index, returns.values * 100, color=col_r, alpha=0.7, width=0.8)
axes[1].axhline(0, color=C['neutral'], lw=0.5)
axes[1].set_ylabel('Retorno (%)')
axes[1].set_title('Retornos diarios', loc='left', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

# Retornos al cuadrado — proxy de volatilidad realizada
axes[2].fill_between(returns.index, (returns**2)*1e4, color=C['garch'], alpha=0.6)
axes[2].set_ylabel('r² × 10⁴')
axes[2].set_xlabel('Fecha')
axes[2].set_title('Retornos² — clusters de volatilidad visibles', loc='left', fontsize=9)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/finance_eda.png')

In [ ]:
# ── TEST ARCH — ¿hay heterocedasticidad condicional? ─────────────────────────
# Test LM de Engle: regresión de r² sobre sus rezagos
# H0: no hay ARCH (coeficientes = 0)
# Si p < 0.05 → usar GARCH

from scipy import stats as sp_stats

def arch_test(series, lags=5):
    r2 = series**2
    n  = len(r2)
    X  = np.column_stack([r2.shift(i).fillna(r2.mean()) for i in range(1, lags+1)])
    y  = r2.values
    # OLS manual
    Xb = np.column_stack([np.ones(n), X])
    b  = np.linalg.lstsq(Xb, y, rcond=None)[0]
    yhat = Xb @ b
    ss_res = ((y - yhat)**2).sum()
    ss_tot = ((y - y.mean())**2).sum()
    r2_stat = 1 - ss_res / ss_tot
    lm  = n * r2_stat
    p   = 1 - sp_stats.chi2.cdf(lm, df=lags)
    return lm, p

lm, p = arch_test(returns, lags=5)
print('── Test ARCH (Engle LM) ──────────────────────────────────────')
print(f'  Estadístico LM : {lm:.2f}')
print(f'  p-valor        : {p:.6f}')
print(f'  Conclusión     : {"✓ ARCH significativo → usar GARCH" if p < 0.05 else "No se detecta ARCH"}')

In [ ]:
# ── GARCH(1,1) — ESTIMACIÓN POR MLE ──────────────────────────────────────────
# Estimamos ω, α, β maximizando la log-verosimilitud gaussiana

def garch_filter(params, r):
    """Filtra la varianza condicional GARCH(1,1). Retorna sigma2 array."""
    omega, alpha, beta = params
    n = len(r)
    sigma2 = np.zeros(n)
    sigma2[0] = np.var(r)
    for t in range(1, n):
        sigma2[t] = omega + alpha * r[t-1]**2 + beta * sigma2[t-1]
    return sigma2

def neg_log_lik(params, r):
    omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
        return 1e10
    sigma2 = garch_filter(params, r)
    sigma2 = np.maximum(sigma2, 1e-12)
    ll = -0.5 * np.sum(np.log(2*np.pi*sigma2) + r**2 / sigma2)
    return -ll

r_arr = returns.values
# Inicializar con varianza muestral
s2_0  = np.var(r_arr)
x0    = [s2_0 * 0.05, 0.08, 0.88]

res = minimize(neg_log_lik, x0, args=(r_arr,),
               method='L-BFGS-B',
               bounds=[(1e-8, None), (1e-4, 0.5), (1e-4, 0.9999)],
               options={'maxiter': 2000})

omega_hat, alpha_hat, beta_hat = res.x
persistence = alpha_hat + beta_hat
sigma2_unconditional = omega_hat / (1 - persistence)

print('── GARCH(1,1) — Parámetros estimados (MLE) ──────────────────')
print(f'  ω (omega)      : {omega_hat:.8f}')
print(f'  α (alpha)      : {alpha_hat:.6f}  ← impacto del shock')
print(f'  β (beta)       : {beta_hat:.6f}  ← persistencia')
print(f'  α + β          : {persistence:.6f}  ← debe ser < 1')
print(f'  σ² largo plazo : {sigma2_unconditional:.8f}')
print(f'  σ largo plazo  : {np.sqrt(sigma2_unconditional):.6f} ({np.sqrt(sigma2_unconditional)*np.sqrt(252):.2%} anual)')
dur = 1 / (1 - beta_hat)
print(f'  Dur. cluster   : ≈{dur:.0f} días para disipar al 63%')

In [ ]:
# ── EGARCH — ESTIMACIÓN ───────────────────────────────────────────────────────

def egarch_filter(params, r):
    omega_e, alpha_e, beta_e, gamma_e = params
    n = len(r)
    log_sigma2 = np.zeros(n)
    log_sigma2[0] = np.log(np.var(r))
    for t in range(1, n):
        z = r[t-1] / np.sqrt(np.exp(log_sigma2[t-1]))
        log_sigma2[t] = (omega_e + beta_e * log_sigma2[t-1]
                         + alpha_e * abs(z) + gamma_e * z)
    return np.exp(log_sigma2)

def neg_ll_egarch(params, r):
    omega_e, alpha_e, beta_e, gamma_e = params
    if abs(beta_e) >= 1:
        return 1e10
    sigma2 = egarch_filter(params, r)
    sigma2 = np.maximum(sigma2, 1e-12)
    ll = -0.5 * np.sum(np.log(2*np.pi*sigma2) + r**2 / sigma2)
    return -ll

x0_e = [-0.2, 0.15, 0.92, -0.05]
res_e = minimize(neg_ll_egarch, x0_e, args=(r_arr,),
                 method='L-BFGS-B',
                 bounds=[(-5,5),(0,1),(-0.999,0.999),(-1,1)],
                 options={'maxiter': 2000})

omega_e, alpha_e, beta_e, gamma_e = res_e.x

print('── EGARCH — Parámetros estimados ────────────────────────────')
print(f'  ω : {omega_e:.6f}')
print(f'  α : {alpha_e:.6f}  ← magnitud del shock')
print(f'  β : {beta_e:.6f}  ← persistencia')
print(f'  γ : {gamma_e:.6f}  ← asimetría '
      f'({"depreciaciones más volátiles" if gamma_e < 0 else "apreciaciones más volátiles"})')

In [ ]:
# ── FILTRAR VARIANZA CONDICIONAL ─────────────────────────────────────────────
sigma2_garch  = garch_filter([omega_hat, alpha_hat, beta_hat], r_arr)
sigma2_egarch = egarch_filter([omega_e, alpha_e, beta_e, gamma_e], r_arr)

df_fin = pd.DataFrame({
    'tc'          : tc.values[:len(returns)],
    'return'      : r_arr,
    'sigma_garch' : np.sqrt(sigma2_garch)  * np.sqrt(252),   # anualizada
    'sigma_egarch': np.sqrt(sigma2_egarch) * np.sqrt(252),
    'sigma_roll20': returns.rolling(20).std() * np.sqrt(252), # Bollinger proxy
}, index=returns.index)

print(f'σ GARCH media   : {df_fin.sigma_garch.mean():.2%}')
print(f'σ EGARCH media  : {df_fin.sigma_egarch.mean():.2%}')
print(f'σ Rolling20 med : {df_fin.sigma_roll20.mean():.2%}')

In [ ]:
# ── FORECAST DE VARIANZA — 30 días adelante ───────────────────────────────────
h = 30
sigma2_unc = omega_hat / (1 - persistence)
sigma2_last = sigma2_garch[-1]
# sigma_{t+1}²
eps_last = r_arr[-1]
sigma2_t1 = omega_hat + alpha_hat * eps_last**2 + beta_hat * sigma2_last

forecast_var = np.array([
    sigma2_unc + persistence**(i-1) * (sigma2_t1 - sigma2_unc)
    for i in range(1, h+1)
])
forecast_vol = np.sqrt(forecast_var) * np.sqrt(252)  # anualizada
forecast_dates = pd.bdate_range(returns.index[-1], periods=h+1)[1:]

print(f'Forecast vol GARCH (próximos {h} días hábiles):')
for i in [1, 5, 10, 20, 30]:
    print(f'  h={i:2d}: σ={forecast_vol[i-1]:.2%}')
print(f'  σ largo plazo: {np.sqrt(sigma2_unc)*np.sqrt(252):.2%}')

In [ ]:
# ── DASHBOARD PRINCIPAL ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 13))
fig.suptitle(
    'GARCH / EGARCH — Volatilidad Condicional PEN/USD\n'
    f'Banco Central de Reserva del Perú · {SOURCE}',
    fontsize=12, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(4, 2, hspace=0.38, wspace=0.28)

# P1 — Tipo de cambio
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df_fin.index, df_fin.tc, color=C['price'], lw=0.9, label='TC PEN/USD')
ax1.set_ylabel('S/. por USD')
ax1.set_title('Panel 1 — Tipo de cambio PEN/USD (BCRP)', loc='left', fontsize=10)
ax1.legend(fontsize=9); ax1.grid(axis='y', alpha=0.3)

# P2 — Volatilidad condicional GARCH vs. EGARCH vs. Rolling
ax2 = fig.add_subplot(gs[1, :])
ax2.plot(df_fin.index, df_fin.sigma_garch * 100,
         color=C['garch'],   lw=0.9,  label='σ GARCH(1,1)')
ax2.plot(df_fin.index, df_fin.sigma_egarch * 100,
         color=C['egarch'],  lw=0.9,  label='σ EGARCH', alpha=0.8)
ax2.plot(df_fin.index, df_fin.sigma_roll20 * 100,
         color=C['neutral'], lw=0.9, ls='--', label='σ Rolling 20d (Bollinger proxy)')
ax2.set_ylabel('Volatilidad anualizada (%)')
ax2.set_title('Panel 2 — σ GARCH vs. EGARCH vs. Rolling 20d', loc='left', fontsize=10)
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# P3 — Retornos estandarizados (residuos GARCH: deberían ser N(0,1))
ax3 = fig.add_subplot(gs[2, 0])
std_resid = r_arr / np.sqrt(sigma2_garch)
ax3.hist(std_resid, bins=60, color=C['fill'],
         edgecolor=C['garch'], lw=0.3, density=True)
x_n = np.linspace(-5, 5, 200)
ax3.plot(x_n, norm.pdf(x_n), color=C['price'], lw=1.2, label='N(0,1)')
ax3.set_title('Panel 3 — Residuos estandarizados\n(deben ser ≈ N(0,1))', loc='left', fontsize=9)
ax3.set_xlabel('z = ε/σ'); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# P4 — Asimetría EGARCH: impacto de shocks + vs −
ax4 = fig.add_subplot(gs[2, 1])
z_range = np.linspace(-4, 4, 200)
impact_pos = alpha_e * np.abs(z_range) + gamma_e * z_range
ax4.plot(z_range, impact_pos, color=C['egarch'], lw=1.5)
ax4.axvline(0, color=C['neutral'], lw=0.7, ls='--')
ax4.axhline(0, color=C['neutral'], lw=0.7, ls='--')
ax4.fill_between(z_range[z_range>=0], impact_pos[z_range>=0],
                 alpha=0.3, color=C['vol'], label='Shock positivo (apreciación)')
ax4.fill_between(z_range[z_range<0], impact_pos[z_range<0],
                 alpha=0.3, color=C['garch'], label='Shock negativo (depreciación)')
ax4.set_title(f'Panel 4 — Asimetría EGARCH\n(γ={gamma_e:.3f})', loc='left', fontsize=9)
ax4.set_xlabel('z (shock estandarizado)'); ax4.set_ylabel('Impacto en ln(σ²)')
ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

# P5 — Forecast de volatilidad
ax5 = fig.add_subplot(gs[3, :])
# Últimos 60 días reales
ax5.plot(df_fin.index[-60:], df_fin.sigma_garch[-60:]*100,
         color=C['garch'], lw=1.2, label='σ GARCH realizada')
ax5.plot(forecast_dates, forecast_vol*100,
         color=C['vol'], lw=1.5, ls='--', label=f'Forecast σ ({h}d)')
ax5.axhline(np.sqrt(sigma2_unc)*np.sqrt(252)*100,
            color=C['neutral'], lw=0.8, ls=':', label='σ largo plazo')
ax5.fill_between(forecast_dates,
                 forecast_vol*100 * 0.85, forecast_vol*100 * 1.15,
                 alpha=0.2, color=C['vol'], label='Intervalo ±15%')
ax5.set_ylabel('Volatilidad anualizada (%)')
ax5.set_xlabel('Fecha')
ax5.set_title(f'Panel 5 — Forecast de volatilidad GARCH ({h} días)', loc='left', fontsize=10)
ax5.legend(fontsize=8); ax5.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/finance_dashboard.png')

In [ ]:
# ── EXPORTAR ─────────────────────────────────────────────────────────────────
df_fin.to_csv('data/finance_garch_output.csv')
pd.DataFrame({'date': forecast_dates,
              'sigma_forecast': forecast_vol}).to_csv('data/finance_vol_forecast.csv', index=False)
print('✓ data/finance_garch_output.csv')
print('✓ data/finance_vol_forecast.csv')
print('✓ data/finance_eda.png')
print('✓ data/finance_dashboard.png')

## Conclusiones — contexto financiero

| Concepto | En PEN/USD | Uso operacional |
|----------|-----------|----------------|
| **ARCH test** | p < 0.05 → clusters confirmados | Validar antes de estimar GARCH |
| **α (impacto)** | Reacción inmediata al shock cambiario | α alto → vol muy reactiva |
| **β (persistencia)** | Duración del cluster de volatilidad | β alto → clusters largos |
| **γ EGARCH** | Depreciaciones más volátiles que apreciaciones | Cobertura asimétrica |
| **Forecast σ** | Vol esperada en las próximas semanas | Sizing de coberturas FX |

**Próximo:** `2_Supply_Adaptation.ipynb` — misma metodología sobre residuos de demanda de Alicorp, donde la varianza condicional determina el Safety Stock semana a semana.